In [ ]:
# Parameters -- Fabric overrides these at pipeline runtime.
keyvault_url = "https://kv-analytically.vault.azure.net/"
xero_env     = ""   # which env's tokens to read: xero-tokens-<env> (dev|prod). PROD passes "prod".
only_org     = ""      # optional Xero tenantId to restrict to; blank = every mapped org
run_uuid     = ""     # correlation id from Orchestrate_Build; blank -> fresh uuid. Progress -> Audit.Ingest_Log


In [ ]:
import json
import time
import base64
import re
from datetime import date, datetime, timezone, timedelta

import requests
from pyspark.sql.types import StringType, StructType, StructField
import notebookutils


In [ ]:
# --- Ingest_Log instrumentation: progress -> Audit.Ingest_Log so a run is watchable in SQL.
# Xero previously logged to stdout ONLY (invisible in the audit trail). Mirrors Ingest_Dentally;
# a log failure never breaks the ingest. Watch live:
#   SELECT Logged_At, Entity, Phase, Rows_Landed, Detail FROM Audit.Ingest_Log
#   WHERE Entity LIKE 'xero%' AND Logged_At > DATEADD(MINUTE,-30,SYSUTCDATETIME()) ORDER BY Logged_At;
import struct, pyodbc, uuid as _uuidlib
try:
    run_uuid
except NameError:
    run_uuid = ""
if not run_uuid:
    run_uuid = str(_uuidlib.uuid4())
_lcur = None
try:
    import sempy.fabric as _fab
    _wsid = _fab.get_workspace_id()
    _whs  = _fab.FabricRestClient().get(f"/v1/workspaces/{_wsid}/warehouses").json()["value"]
    _ep   = next((w["properties"]["connectionString"] for w in _whs if w["displayName"] == "WH_Dentally"), None)
    _tb   = notebookutils.credentials.getToken("https://database.windows.net/").encode("UTF-16-LE")
    _ts   = struct.pack(f"<I{len(_tb)}s", len(_tb), _tb)
    _lconn = pyodbc.connect("Driver={ODBC Driver 18 for SQL Server};Server=" + _ep + ",1433;Database=WH_Dentally;Encrypt=yes;TrustServerCertificate=no;", attrs_before={1256: _ts})
    _lconn.autocommit = True
    _lcur = _lconn.cursor()
    print("ingest-log -> Audit.Ingest_Log (run_uuid " + run_uuid + ")")
except Exception as _e:
    print("ingest-log DISABLED (cell output only):", str(_e)[:200])

def log_ingest(entity, phase, rows=None, tenant_id=None, detail=None):
    print("  [" + str(phase) + "] " + str(entity) + (" rows=" + str(rows) if rows is not None else "") + ((" " + str(detail)) if detail else ""))
    if _lcur is None:
        return
    try:
        _lcur.execute(
            "INSERT INTO Audit.Ingest_Log (Run_UUID, Tenant_ID, Entity, Phase, Rows_Landed, Detail, Logged_At) VALUES (?,?,?,?,?,?,SYSUTCDATETIME())",
            run_uuid, (int(tenant_id) if tenant_id is not None else None), str(entity)[:100], str(phase)[:20], rows, (str(detail)[:1000] if detail is not None else None))
    except Exception:
        pass

WM_OVERLAP_HOURS = 4

def bronze_since(tenant_id, source):
    # Approved incremental pattern: derive the per-table (per Xero Source) high-watermark FROM
    # BRONZE itself = MAX(DW_Loaded_At) for this tenant+source, minus a 4h overlap (the idempotent
    # doc-merge makes the overlap harmless). None -> COLD full pull (empty Bronze self-bootstraps).
    # No bespoke watermark table: Bronze IS the watermark and the doc-merge advances DW_Loaded_At
    # each run, so a low-activity org keeps a tight If-Modified-Since window (not months of re-pull).
    if _lcur is None:
        return None
    try:
        _lcur.execute("SELECT MAX(DW_Loaded_At) FROM Bronze.Xero_Lines WHERE Tenant_ID = ? AND Source = ?", int(tenant_id), source)
        row = _lcur.fetchone()
        if row and row[0] is not None:
            return row[0] - timedelta(hours=WM_OVERLAP_HOURS)
    except Exception as _e:
        print("  bronze watermark read failed " + str(source) + ": " + str(_e)[:120])
    return None

log_ingest("xero_ingest", "START", detail="env=" + str(xero_env))

In [ ]:
# --- Key Vault I/O -----------------------------------------------------------
# Secrets used (tokens + org map are per-env: dev(Demo) and prod(clients) are isolated):
#   xero-client-id, xero-client-secret : the Xero app credentials (shared)
#   xero-org-map-<env> : JSON {"<xeroTenantId>": {"tenant_id": N, "default_site_id": "S"}}
#   xero-tokens-<env>  : JSON {"<connKey>": {"tokens": {...}, "tenants": [...]}} -- one
#                        entry per OAuth consent; refresh tokens rotate and are written back.
# The run identity (pipeline/workspace) needs secrets get + set on the vault.

def kv_get(name):
    return notebookutils.credentials.getSecret(keyvault_url, name)

def kv_set(name, value):
    # notebookutils has no setSecret; write via the Key Vault REST data plane with an
    # AAD token for the vault. If "keyvault" errors as an audience, try "vault" or
    # "https://vault.azure.net". Requires secret 'set' permission for the run identity.
    token = notebookutils.credentials.getToken("keyvault")
    resp = requests.put(
        keyvault_url.rstrip("/") + "/secrets/" + name + "?api-version=7.4",
        headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"},
        json={"value": value},
        timeout=60,
    )
    resp.raise_for_status()


In [ ]:
XERO_CLIENT_ID     = kv_get("xero-client-id")
XERO_CLIENT_SECRET = kv_get("xero-client-secret")
ORG_MAP = json.loads(kv_get("xero-org-map-" + xero_env))
TOKENS  = json.loads(kv_get("xero-tokens-" + xero_env))   # env-isolated: dev(Demo) vs prod(clients)

TOKEN_URL = "https://identity.xero.com/connect/token"
API_BASE  = "https://api.xero.com/api.xro/2.0"
PAGE_SIZE = 100
MAX_429   = 6

def basic_auth():
    raw = (XERO_CLIENT_ID + ":" + XERO_CLIENT_SECRET).encode()
    return "Basic " + base64.b64encode(raw).decode()

def resolve_org(tenant):
    entry = ORG_MAP.get(tenant.get("tenantId"))
    if entry:
        return entry["tenant_id"], entry.get("default_site_id")
    if "demo company" in (tenant.get("tenantName", "") or "").lower():
        return 99, None
    return None, None


In [ ]:
class XeroThrottled(Exception):
    def __init__(self, problem, retry_after):
        self.problem = problem
        self.retry_after = retry_after
        super().__init__("Xero throttled (" + str(problem) + ", Retry-After=" + str(retry_after) + "s)")

SLEEP_CAP = 90   # wait out a short (minute) throttle; a longer/daily one -> raise XeroThrottled (skip org)

def refresh(blob):
    r = requests.post(TOKEN_URL,
        headers={"Authorization": basic_auth(),
                 "Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "refresh_token",
              "refresh_token": blob["tokens"]["refresh_token"]}, timeout=60)
    r.raise_for_status()
    blob["tokens"].update(r.json())
    return blob["tokens"]["access_token"]

def xget(path, access, tid, params=None, modified_since=None):
    _hdr = {"Authorization": "Bearer " + access, "Xero-tenant-id": tid, "Accept": "application/json"}
    if modified_since is not None:
        _hdr["If-Modified-Since"] = modified_since.strftime("%Y-%m-%dT%H:%M:%S")
    for _ in range(MAX_429):
        r = requests.get(API_BASE + path, headers=_hdr, params=params or {}, timeout=(10, 60))
        if r.status_code == 429:
            retry_after = int(r.headers.get("Retry-After", 5))
            problem = r.headers.get("X-Rate-Limit-Problem", "") or ""
            if "log_ingest" in globals():
                log_ingest("xero" + path.replace("/", "_").lower(), "THROTTLED",
                           detail=problem + " Retry-After=" + str(retry_after) + "s")
            # Short minute/app throttle: wait it out. A daily cap or a long wait: skip this org
            # (its per-tenant quota is spent) rather than sleeping for hours.
            if retry_after <= SLEEP_CAP and problem.lower() != "daily":
                print("      429", problem, "; sleeping", retry_after + 1, "s")
                time.sleep(retry_after + 1)
                continue
            raise XeroThrottled(problem or "long", retry_after)
        r.raise_for_status()
        return r.json()
    raise XeroThrottled("repeated", None)

def xget_all(path, key, access, tid, modified_since=None):
    out, page = [], 1
    while True:
        rows = xget(path, access, tid, {"page": page}, modified_since=modified_since).get(key, []) or []
        out.extend(rows)
        if len(rows) < PAGE_SIZE:
            return out
        page += 1

In [ ]:
def parse_date(v):
    if not v:
        return None
    m = re.search(r"/Date\((\d+)", str(v))
    if m:
        return datetime.fromtimestamp(int(m.group(1)) / 1000, tz=timezone.utc).date().isoformat()
    return str(v)[:10]

def tracking_pairs(tracking):
    pairs = [(t.get("Name"), t.get("Option")) for t in (tracking or [])]
    cols = {}
    for i in range(2):
        cat, opt = pairs[i] if i < len(pairs) else (None, None)
        cols["Tracking_Cat_" + str(i + 1)] = cat
        cols["Tracking_Opt_" + str(i + 1)] = opt
    return cols

def flatten_lineitems(items, source, id_key, num_key, tenant, xtid):
    rows = []
    for it in items:
        base = {"Tenant_ID": tenant, "Xero_Tenant_ID": xtid, "Source": source,
                "Doc_ID": it.get(id_key), "Doc_Number": it.get(num_key),
                "Doc_Type": it.get("Type"), "Doc_Status": it.get("Status"),
                "Doc_Date": parse_date(it.get("DateString") or it.get("Date")),
                "Contact_Name": (it.get("Contact") or {}).get("Name"),
                "Line_Amount_Types": it.get("LineAmountTypes")}
        for ln in it.get("LineItems", []):
            row = dict(base)
            row.update({"Line_Item_ID": ln.get("LineItemID"),
                        "Account_Code": ln.get("AccountCode"),
                        "Account_ID": ln.get("AccountID"),
                        "Description": ln.get("Description"),
                        "Line_Amount": ln.get("LineAmount"),
                        "Tax_Amount": ln.get("TaxAmount"),
                        "Tracking": ln.get("Tracking")})
            row.update(tracking_pairs(ln.get("Tracking")))
            rows.append(row)
    return rows

def flatten_manual_journals(items, tenant, xtid):
    rows = []
    for mj in items:
        base = {"Tenant_ID": tenant, "Xero_Tenant_ID": xtid, "Source": "MANUALJOURNAL",
                "Doc_ID": mj.get("ManualJournalID"), "Doc_Number": None,
                "Doc_Type": "MANJRNL", "Doc_Status": mj.get("Status"),
                "Doc_Date": parse_date(mj.get("Date")),
                "Contact_Name": None, "Line_Amount_Types": mj.get("LineAmountTypes")}
        for jl in mj.get("JournalLines", []):
            row = dict(base)
            row.update({"Line_Item_ID": None,
                        "Account_Code": jl.get("AccountCode"),
                        "Account_ID": jl.get("AccountID"),
                        "Description": jl.get("Description"),
                        "Line_Amount": jl.get("LineAmount"),
                        "Tax_Amount": None,
                        "Tracking": jl.get("Tracking")})
            row.update(tracking_pairs(jl.get("Tracking")))
            rows.append(row)
    return rows

def flatten_tracking(cats, tenant, xtid):
    rows = []
    for c in cats:
        for opt in c.get("Options", []):
            rows.append({"Tenant_ID": tenant, "Xero_Tenant_ID": xtid,
                         "Tracking_Category_ID": c.get("TrackingCategoryID"),
                         "Category_Name": c.get("Name"), "Category_Status": c.get("Status"),
                         "Tracking_Option_ID": opt.get("TrackingOptionID"),
                         "Option_Name": opt.get("Name"), "Option_Status": opt.get("Status")})
    return rows


In [ ]:
def to_str(v):
    if v is None:
        return None
    if isinstance(v, (dict, list)):
        return json.dumps(v)
    return str(v)

def write_stage(records, table_name, tenant):
    full = "stage_" + table_name
    if not records:
        print("  " + table_name + ": 0 rows")
        log_ingest(table_name, "EMPTY", rows=0, tenant_id=tenant)
        return
    keys = set()
    for r in records:
        keys.update(r.keys())
    schema = StructType([StructField(k, StringType(), True) for k in sorted(keys)])
    str_records = [{k: to_str(r.get(k)) for k in keys} for r in records]
    df = spark.createDataFrame(str_records, schema=schema)
    if spark.catalog.tableExists(full):
        # Overwrite only this tenant's rows; other tenants are preserved. mergeSchema
        # lets new columns (e.g. Tracking_Cat_1) appear without failing.
        df.write.format("delta").mode("overwrite") \
            .option("replaceWhere", "Tenant_ID = '" + str(tenant) + "'") \
            .option("mergeSchema", "true").saveAsTable(full)
    else:
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(full)
    print("  " + table_name + ": " + str(len(records)) + " rows -> " + full)
    log_ingest(table_name, "WROTE", rows=len(records), tenant_id=tenant)


In [ ]:
# Phase 1: refresh every connection FIRST and persist the rotated refresh tokens,
# so a later extraction failure can't strand a rotated (now-invalid) token.
log_ingest("xero_refresh", "START", rows=len(TOKENS))
access_by_conn = {}
for conn_key, blob in TOKENS.items():
    log_ingest("xero_refresh", "CONN", detail=str(conn_key))
    access_by_conn[conn_key] = refresh(blob)
kv_set("xero-tokens-" + xero_env, json.dumps(TOKENS))
log_ingest("xero_refresh", "DONE", rows=len(TOKENS))
print("Refreshed", len(TOKENS), "connection(s); tokens persisted.")

def _extract_org(tenant, access, xtid, tenant_id, default_site):
    write_stage([{"Tenant_ID": tenant_id, "Xero_Tenant_ID": xtid,
                  "Tenant_Name": tenant.get("tenantName"),
                  "Default_Site_ID": default_site}], "xero_orgs", tenant_id)

    log_ingest("xero_accounts", "FETCH", tenant_id=tenant_id)
    accounts = xget("/Accounts", access, xtid)["Accounts"]
    acc_rows = [{"Tenant_ID": tenant_id, "Xero_Tenant_ID": xtid,
                 "Account_ID": a.get("AccountID"), "Code": a.get("Code"),
                 "Name": a.get("Name"), "Type": a.get("Type"), "Class": a.get("Class"),
                 "Reporting_Code": a.get("ReportingCode"),
                 "Reporting_Code_Name": a.get("ReportingCodeName"),
                 "Status": a.get("Status")} for a in accounts]
    write_stage(acc_rows, "xero_accounts", tenant_id)

    log_ingest("xero_tracking", "FETCH", tenant_id=tenant_id)
    tracking = xget("/TrackingCategories", access, xtid).get("TrackingCategories", []) or []
    write_stage(flatten_tracking(tracking, tenant_id, xtid), "xero_tracking", tenant_id)

    lines = []
    _line_eps = [("Invoices", "Invoices", "INVOICE", "InvoiceID", "InvoiceNumber"),
                 ("CreditNotes", "CreditNotes", "CREDITNOTE", "CreditNoteID", "CreditNoteNumber"),
                 ("BankTransactions", "BankTransactions", "BANK", "BankTransactionID", "Reference")]
    for _path, _key, _source, _idk, _numk in _line_eps:
        _since = bronze_since(tenant_id, _source)
        log_ingest("xero_lines", "FETCH", tenant_id=tenant_id, detail=_path + " " + ("since " + _since.strftime("%Y-%m-%d %H:%M") if _since else "COLD full"))
        lines += flatten_lineitems(xget_all("/" + _path, _key, access, xtid, modified_since=_since),
                                   _source, _idk, _numk, tenant_id, xtid)
    _mj_since = bronze_since(tenant_id, "MANUALJOURNAL")
    log_ingest("xero_lines", "FETCH", tenant_id=tenant_id, detail="ManualJournals " + ("since " + _mj_since.strftime("%Y-%m-%d %H:%M") if _mj_since else "COLD full"))
    lines += flatten_manual_journals(xget_all("/ManualJournals", "ManualJournals", access, xtid, modified_since=_mj_since),
                                     tenant_id, xtid)
    write_stage(lines, "xero_lines", tenant_id)

# Phase 2: extract + land per mapped org. Xero's minute/daily limits are PER TENANT, so a
# throttle on one org skips THAT org (logged ORG_THROTTLED) and continues the rest -- it never
# sleeps for hours and never blocks the other orgs.
processed = 0
throttled = 0
for conn_key, blob in TOKENS.items():
    access = access_by_conn[conn_key]
    for tenant in blob.get("tenants", []):
        xtid = tenant["tenantId"]
        if only_org and xtid != only_org:
            continue
        tenant_id, default_site = resolve_org(tenant)
        if tenant_id is None:
            log_ingest(tenant.get("tenantName") or xtid, "SKIP_UNMAPPED", detail=str(xtid))
            print("Skip unmapped org:", tenant.get("tenantName"), xtid)
            continue
        log_ingest(tenant.get("tenantName") or xtid, "ORG_START", tenant_id=tenant_id, detail="xtid=" + str(xtid) + " (per-source Bronze watermark)")
        print("Org:", tenant.get("tenantName"), "-> Tenant_ID", tenant_id)
        try:
            _extract_org(tenant, access, xtid, tenant_id, default_site)
            processed += 1
        except XeroThrottled as _t:
            throttled += 1
            log_ingest(tenant.get("tenantName") or xtid, "ORG_THROTTLED", tenant_id=tenant_id,
                       detail=str(_t.problem) + " Retry-After=" + str(_t.retry_after) + "s -- skipped; other orgs continue")
            print("Org throttled -> skipping to next:", tenant.get("tenantName"), _t)
            continue

log_ingest("xero_ingest", "COMPLETE", rows=processed, detail=("throttled=" + str(throttled) if throttled else None))
print("Done. Extracted", processed, "org(s); throttled", throttled, ". Bronze/Silver/Gold Xero loads run next in the pipeline.")